## Environment Setup

In [ ]:
import os
import json
import xpd_tools
from tkinter.constants import N
import pandas as pd
import numpy as np

## Materials Project

Get some data from Materials Project to run. We will interpolate between the points adn use it as a sample target for BLOP

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
MP_API_KEY = os.environ["MP_API_KEY"]


## AGENT - Initialization

### Initialize the Agent and set the logical paths and globals of our system.

For Bluesky:
- Queueserver toggle
- Tiled profile

Other:
- HTTP API and Server uri's
- ZMQ consumer addresses

Evaluation methods:
- xray, uvvis, or xray-uvis


In [ ]:
from xpd_tools.optimization.agent import BuildAgent

# Queueserver connection -- unused on this build_local() path, kept here
# only because HTTP_SERVER_URI/HTTP_API_KEY/ZMQ_CONSUMER_ADDR are read from
# the environment below regardless.
HTTP_SERVER_URI = os.environ.get(
    "QSERVER_HTTP_URI", 
    "https://xf28id2-xpd-qs1.nsls2.bnl.gov"
    )
HTTP_API_KEY = os.environ.get(
    "QSERVER_HTTP_SERVER_API_KEY", 
    ""
    )
ZMQ_CONSUMER_ADDR = os.environ.get(
    "ZMQ_CONSUMER_ADDR",
    "ipc:///var/lib/bluesky-zmq-proxy/xpd-ipc-in-ipc-out/out.sock",
    )

# Historical data path
AGENT_DATA_PATH = "agent_halide_data_uvvis_only_fake.csv"
# AGENT_DATA_PATH = None

# Tiled URI for evaluation function
TILED_URI = os.environ.get(
    "TILED_URI", 
    "https://tiled.nsls2.bnl.gov"
    )
TILED_PROFILE = os.environ.get(
    "TILED_PROFILE", 
    "xpd"
    )

build_agent = BuildAgent(
    # False -- this notebook exercises build_local(), the no-queue-server
    # path (drives a local RunEngine directly instead of dispatching
    # acquisition plans by name through a Queue Server). build() (the
    # queue_server=True path) is what examples/halide_example_class.ipynb
    # demonstrates instead.
    queue_server = False,
    # Agent
    agent_data_path = AGENT_DATA_PATH,
    # Http 
    http_server_uri = HTTP_SERVER_URI,
    http_api_key = HTTP_API_KEY,
    # ZMQ
    zmq_consumer_address = ZMQ_CONSUMER_ADDR,
    # Tiled
    tiled_profile = TILED_PROFILE,
    # Evaluation Method
    evaluation_method = 'xray-uvvis'
)


# Provide the agent some metadata
build_agent.set_metadata({
    "beamline": "28id2",
    "tags": ["qserver", "bluesky"],
    "comment": "Halide Synthesis Test"
    })
print(build_agent.metadata_string)

## BEAMLINE - PDF Xray Diffraction

Objective correlation function with the phases that will be tested.


Correlation functions:
- pearson, nn-matrix, weighted-profile-r, cross-correlation, or ensemble

In [ ]:
from xpd_tools.optimization.helpers.phases import Phase

# PDF correlation function
PDF_FUNCTION = "pearson"

# Set the phase selection objectives and directions
build_agent.set_xray_objectives(
    # Measurement
    max_retries     = 10,
    retry_delay     = 2.0,
    # Configuration
    exposure        = 6.0,
    frame_acq_time  = 1.5,
    no_dark         = False,
    stream_name     = "scattering",
    # Screening: UV-Vis is measured (screen_only/screen_and_record) or not
    # (unscreened). We configure full UV-Vis objectives below, so we want
    # the measurement recorded, not just used to gate acquisition.
    screening       = "screen_and_record",
    # Quality Checks
    use_good_bad    = True,
    good_target     = 2,
    max_bad         = 3,
    num_abs         = 16,
    num_flu         = 16,
    # PDF correlation masking window (Angstroms)
    min_radius      = 2.0,
    max_radius      = 20.0,
    # Phase Fitting
    objective_function = PDF_FUNCTION,
    phases   = [
        Phase(
            name      = "CsPbBr3",
            gr        = "refdata/CsPbBr3.gr",
            cif       = "refdata/CsPbBr3.cif",
            minimize  = False
            ),
        Phase(
            name      = "CsPbI3",
            gr        = "refdata/CsPbI3.gr",
            cif       = "refdata/CsPbI3.cif",
            minimize  = False
            ),
        Phase(
            name      = "CsPbCl3",
            gr        = "refdata/CsPbCl3.gr",
            cif       = "refdata/CsPbCl3.cif",
            minimize  = False
            ),
        Phase(
            name      = "CsBr",
            gr        = "refdata/CsBr.gr",
            cif       = "refdata/CsBr.cif",
            minimize  = True
            ),
        Phase(
            name      = "Cs4PbBr6",
            gr        = "refdata/Cs4PbBr6.gr",
            cif       = "refdata/Cs4PbBr6.cif",
            minimize  = True
            ),
    ]
  )
print(build_agent.xray_settings)
print(build_agent.quality_policy)
print("screening:", build_agent.screening)
print("min/max radius:", build_agent.min_radius, build_agent.max_radius)

## BEAMLINE - UVvis Screening

In [ ]:
# Optimization Parameters
PEAK_TARGET         = 450   # nm
PEAK_TOLERANCE      = 5     # nm

from xpd_tools.optimization.helpers.qepro import PlqyReference, SpectraFitSettings

build_agent.set_uvvis_objectives(
    # Objective Target
    peak_target             = PEAK_TARGET,
    peak_tolerance          = PEAK_TOLERANCE,
    max_retries             = 10,
    retry_delay             = 2.0,
    # Screening, data selection, and fitting windows
    fit_settings = SpectraFitSettings(
        pl_screen_key_height        = 200,
        pl_screen_peak_height       = 50,
        pl_screen_peak_distance     = 100,
        pl_percent_range            = (40, 100),
        pl_wavelength_range         = (400, 800),
        pl_fit_maxfev               = 100000,
        pl_fit_r2_window_sigma      = 3,
        absorbance_percent_range    = (10, 70),
        absorbance_wavelength_range = (210, 700),
    ),
    # Calibration Standard Reference (reference_type defaults to "quinine")
    plqy = PlqyReference(
        excitation_wavelength_nm   = 365,
        absorbance                 = 0.361,
        pl_integral                = 952628,
        refractive_index           = 1.337,
        plqy                       = 0.546,
        solvent_refractive_index   = 1.506,
    ),
)

print(build_agent.fit_settings)
print(build_agent.plqy)

## BLOP - DOFs

Degrees of freedom that BLOP can modify

In [ ]:
# Set the Agent DOFs
from xpd_tools.optimization.helpers.dofs import Pump

build_agent.set_dofs(
    pumps = [
      Pump(
        name = "CsPb",
        bounds = (10, 200),
        id = "dds2_p1"
      ),
      Pump(
        name = "Br",
        bounds = (5, 200),
        id = "dds2_p2"
      ),
      Pump(
        name = "Cl",
        bounds = (0, 200),
        id = "dds3_p1"
      ),
      Pump(
        name = "I",
        bounds = (0, 200),
        id = "dds3_p2"
      )
    ]
  )
print(build_agent.dofs)

## BEAMLINE - Experimental Harware Parameters

In [ ]:
from xpd_tools.optimization.plans import FlowSource, DilutionStage, WashCycle

# loaded_ml: mL filled into each syringe (placeholder = target_ml; set on the day)
build_agent.experiment(
    sources = [
        FlowSource(
            dof="infusion_rate_CsPb",
            pump='dds2_p1',
            precursor="CsPbOA",
            sample_label="CsPb",
            loaded_ml=30.0,
        ),
        FlowSource(
            dof="infusion_rate_Br",
            pump='dds2_p2',
            precursor="TOABr",
            sample_label="Br",
            loaded_ml=30.0,
        ),
        FlowSource(
            dof="infusion_rate_Cl",
            pump='dds3_p1',
            precursor="ZnCl2",
            sample_label="Cl",
            loaded_ml=30.0,
        ),
        FlowSource(
            dof="infusion_rate_I",
            pump='dds3_p2',
            precursor="I2",
            sample_label="I",
            loaded_ml=30.0,
        ),
    ],
    dilutions = [
        DilutionStage(
            pump='dds1_p1',
            ratio=1.0,
            position="before_equilibrium",
            syringe_ml=20,
            material="plastic_BD",
            target_ml=20,
            loaded_ml=20.0,
        ),
        DilutionStage(
            pump='ultra2',
            ratio=1.0,
            position="after_equilibrium",
            syringe_ml=100,
            material="steel",
            target_ml=100,
            loaded_ml=100.0,
            wait_sec=30,
        ),
    ],
    wash_cycles=[
        WashCycle(pump='ultra1', loaded_ml=30.0),
    ]
)

## BLOP - Historical data and run settings

In [ ]:
# Historical data (AGENT_DATA_PATH, above) is loaded by build()/build_local():
# peak_distance is recomputed from Peak with the configured peak_target, and rows
# with NaN/inf are dropped (an error if that leaves none). How the campaign runs
# is the agent's run settings: to_config() saves them as the JSON's "run" section,
# and from_config() / xpd-autonomous-server read them back.
RUN = dict(
    iterations=30,
    extra_initialization_trials=5,  # initialization_budget = historical trials + this
    generation_strategy={
        "initialize_with_center": False,
        "min_observed_initialization_trials": 2,
        "allow_exceeding_initialization_budget": True,
    },
)
build_agent.set_run(**RUN)
print(build_agent.run_settings)

## BLOP - Build and Run Agent

In [ ]:
from bluesky.run_engine import RunEngine
from xpd_tools.optimization.simulation import (
    build_simulated_tiled_clients,
    linear_dof_weights,
)
from xpd_tools.optimization.legacy import (
    build_xpd_objects
)

# One DOF per phase: which rate most drives that phase forming.
# A_n B X_(2+n)
# n = 1, 3D perovskite structure (CsPbI3, CsPbCl3, CsPbBr3)
# (n=2-4 being low dimensional)
# n = 2, 2D structure 
# n = 3, chains
# n = 4, 0D or zero dimensional
DOF_FOR_PHASE = {
    # 3D strucutreshttps://doi.org/10.1021/nl5048779
    "CsPbI3": "infusion_rate_I", # 3-D
    "CsPbBr3": "infusion_rate_Br", # 3-D
    "CsPbCl3": "infusion_rate_Cl", # 3-D
}

# Per-phase PL width/absorbance -- toy placeholders, mixed by the same weights as the band gap.
PL_FWHM_NM_FOR_PHASE = {
    "CsBr": 15.0,
    #
    "CsPbCl3": 12.0,
    "CsPbBr3": 20.0,
    "CsPbI3": 42.0,
    #
    "Cs4PbBr6": 25.0,
}
ABSORBANCE_FOR_PHASE = {
    "CsBr": 0.2,
    #
    "CsPbCl3": 0.5,
    "CsPbBr3": 0.4,
    "CsPbI3": 0.3,
    #
    "Cs4PbBr6": 0.4,
}

# Clamp for 1240/mixed_band_gap_eV -- classify_pl hardcodes a >=400nm
# floor (treats shorter as an LED artifact), and CsBr's real (ionic salt)
# band gap alone can push the mixed peak well below that. Keep a 10nm
# margin above the floor and below fit_settings' pl_wavelength_range=(400,800).
PEAK_NM_RANGE = (410.0, 780.0)

# Only CsPbBr3 (the wanted phase) is PL-emissive -- CsBr/Cs4PbBr6 are dark and shouldn't pull the PL peak.
PL_PHASES = ("CsPbCl3","CsPbBr3","CsPbI3")

# Shared across scattering/PL/absorbance -- small relative to absorbance's ~0.2-0.4 scale, negligible for the others.
NOISE_LEVEL = 0.01

RE = RunEngine()

# The simulation settings are part of the run settings, so the saved JSON (and
# the GUI) simulate exactly this. The MP API key stays out: it's read from MP_API_KEY.
build_agent.set_run(
    **RUN,
    local={
        "simulated": {
            "dof_for_phase": DOF_FOR_PHASE,
            "pl_fwhm_nm": PL_FWHM_NM_FOR_PHASE,
            "pl_phases": list(PL_PHASES),
            "absorbance_value": ABSORBANCE_FOR_PHASE,
            "peak_nm_range": list(PEAK_NM_RANGE),
            "noise_level": NOISE_LEVEL,
        },
    },
)
tiled_client, sandbox_client = build_simulated_tiled_clients(
    RE,
    phases=build_agent.phases,
    mp_api_key=MP_API_KEY,
    **build_agent.run_settings.local.simulated,
)


In [ ]:
from xpd_tools.optimization.legacy import (
    identity_wrap_xray_run
)

run_agent = build_agent.build_local(
    devices=build_xpd_objects(),
    wrap_xray_run=identity_wrap_xray_run,
    # Real hardware defaults (30cm mixer, ratio=1.0) compute a real
    # multi-minute equilibrium wait from the pump rate -- zero them out
    # for a fast local/simulated run.
    mixer_lengths_cm=build_agent.run_settings.local.mixer_lengths_cm,
    residence_time_ratio=build_agent.run_settings.local.residence_time_ratio,
    tiled_client=tiled_client,
    sandbox_client=sandbox_client,
)

In [ ]:
# Generation strategy from the run settings (the same call xpd-autonomous-server
# makes): initialization_budget = historical trials + extra_initialization_trials.
n_historical = build_agent.configure_generation_strategy(run_agent)
print(f"{n_historical} historical trials ingested")

In [ ]:
# Save the config, including its "run" section, for xpd-autonomous-server.
build_agent.to_config("halide_example_cubics_with_historical_data.json")

### BLOP - Run

In [ ]:
RE(run_agent.optimize(iterations=build_agent.run_settings.iterations, n_points=build_agent.run_settings.n_points))

## Summarize and Export Data

In [ ]:
from pathlib import Path

Path("tmp/output").mkdir(parents=True, exist_ok=True)
df = run_agent.ax_client.summarize()
df.to_csv("tmp/output/agent_halide_data_cubics.csv")